<a href="https://colab.research.google.com/github/Kirrrk-git/rise-unet-rzsm/blob/mindanao-adaptation/notebooks/06_mindanao_datacube_and_anomaly_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# RISE-UNet Step 21D.4: 11-Year Production RZSM Data Cube Compilation & Anomaly Pipeline

**Authoritative Parent Study**: Lesinger & Tian (2025), *Nature Communications*, DOI: `10.1038/s41467-025-62761-3`  
**Target Baseline Model**: **Mindanao Model A0** (Adapted from EX29 Recursive Hybrid RISE-UNet)  
**Git Branch**: `mindanao-adaptation`  
**Milestone**: Sub-Phase 21D (Step 21D.4-PREFLIGHT & Step 21D.4 Production Cube Compilation)  

### Production Pipeline Objectives:
1. **Step 21D.4-PREFLIGHT (Preflight Verification Gate)**: Audit all 265 NetCDF files covering the 12 Dec 2014–31 Dec 2025 support window ($96,912$ hourly timestamps across $4,038$ calendar days), verifying complete retention of layers `swvl1`, `swvl2`, `swvl3`, leap Februaries, and strict alignment with the frozen spatial contract.
2. **Depth-Weighted Integration & Land-Aware Remapping**: Calculate the 0–100 cm volumetric index ($\text{RZSM}_{0-100} = 0.07\cdot\text{SM}_1 + 0.21\cdot\text{SM}_2 + 0.72\cdot\text{SM}_3$) and remap to Candidate A $0.25^\circ$ grid ($32 \times 48$) via high-performance vectorized land-aware remapping with nearest-neighbor coastal fallback.
3. **Continuous Backward Trailing Rolling Mean**: Compute continuous 7-day backward trailing rolling average (`center=False`, zero future leakage) over the full $4,038$-day archive series, ensuring early 2015 cases have complete trailing rolling memory.
4. **Locked Model A0 Seasonal Climatology & Anomalies**: Fit 3-month seasonal climatology (`season`: DJF, MAM, JJA, SON) strictly on nominal training years $2015 \le \text{year} \le 2021$, and derive daily seasonal anomalies.
5. **Domain-Wide Active Scalar Normalization**: Fit min-max normalization bounds strictly over active evaluation cells ($M_{i,j}=1$) within the training period, standardizing anomalies to $[0, 1]$ while strictly zero-filling the 1,410 non-evaluation cells.
6. **Nominal Timeline Slicing**: Slice nominal production timeline: 01 Jan 2015 to 31 Dec 2025 ($4,018$ days; $506,268$ evaluation cell-days), preserving 2014 exclusively as antecedent support memory.
7. **Census Audit & Zero-NaN Certification**: Assert strictly Zero NaNs/Infs across all 506,268 nominal active evaluation points.
8. **CF-1.8 NetCDF Export & Cloud Lake Sync**: Export final production cube `era5_land_rzsm_production_2015_2025.nc` to `gs://rise-unet-rzsm/processed/rzsm/production/`.


### Step 0: Google Colab Setup & Runtime Initialization
Configures high-speed cloud runtime, detects Colab execution, clones active branch `mindanao-adaptation`, authenticates Google Cloud Storage access, and installs dependencies.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('--> Running in Google Colab environment.')
    from google.colab import auth
    auth.authenticate_user()
    
    # Clone repository or pull latest commits
    repo_path = Path('/content/rise-unet-rzsm')
    if not repo_path.exists():
        print('--> Cloning repository (branch: mindanao-adaptation)...')
        subprocess.run(['git', 'clone', '-b', 'mindanao-adaptation', '--depth', '1',
                        'https://github.com/Kirrrk-git/rise-unet-rzsm.git', str(repo_path)], check=True)
    else:
        print('--> Pulling latest repository commits from origin/mindanao-adaptation...')
        subprocess.run(['git', '-C', str(repo_path), 'pull', 'origin', 'mindanao-adaptation'], check=True)
        
    os.chdir(str(repo_path))
    sys.path.insert(0, str(repo_path))
    
    # Install required high-performance dependencies
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'xarray', 'netCDF4', 'scipy', 'gcsfs', 'matplotlib', 'pyyaml'], check=True)
else:
    print('--> Running in local workstation environment.')
    repo_path = Path.cwd()
    sys.path.insert(0, str(repo_path))

print(f'--> Working directory: {Path.cwd()}')


### Step 0b: Spatial Foundation Contract Verification
Assert the existence and integrity of the 5 frozen spatial foundation artifacts establishing Candidate A ($32 \times 48$) and the 126-cell binary evaluation mask ($f \ge 0.50$).


In [ ]:
import numpy as np
import xarray as xr
import yaml

contract_path = Path('contracts/spatial/spatial_grid_contract.yaml')
assert contract_path.exists(), f'Spatial contract missing at {contract_path}'

with open(contract_path, 'r') as f:
    spatial_contract = yaml.safe_load(f)

grid_nc = Path('processed/grid/mindanao_025deg.nc')
mask_nc = Path('processed/grid/mindanao_eval_mask_025.nc')
assert grid_nc.exists(), f'Grid NC missing at {grid_nc}'
assert mask_nc.exists(), f'Mask NC missing at {mask_nc}'

ds_grid = xr.open_dataset(grid_nc)
ds_mask = xr.open_dataset(mask_nc)

eval_mask = ds_mask['evaluation_mask'].values
n_eval = int(np.sum(eval_mask == 1))
n_buffer = int(np.sum(eval_mask == 0))

print(f'[PASS] Spatial grid verified: {ds_grid.dims["lat"]} lats x {ds_grid.dims["lon"]} lons (32 x 48)')
print(f'[PASS] Evaluation cells: {n_eval} active (expected 126), {n_buffer} zero-filled buffer (expected 1,410)')
assert n_eval == 126, f'Expected 126 evaluation cells, got {n_eval}'
assert n_buffer == 1410, f'Expected 1,410 buffer cells, got {n_buffer}'


### Step 1: Step 21D.4-PREFLIGHT (Production Preflight Verification Gate)
Executes the formal 11-point Preflight Gate before running large-scale tensor compilations:
- 1. Input File Census = Exactly 265 NetCDF files in `gs://rise-unet-rzsm/raw/era5_land/production/`
- 2. Temporal Coverage = 12 Dec 2014 to 31 Dec 2025 ($4,038$ continuous archive calendar days)
- 3. Variable Completeness = `swvl1`, `swvl2`, `swvl3` present across all archive files
- 4. Hourly Timestamp Continuity = 24 hourly records/day; exactly $96,912$ hourly timestamps across support period; leap days intact
- 5. Timestamp Hygiene = Monotonic time coordinate with zero duplicate timestamps
- 6. Grid Geometry Compatibility = Candidate A cell centers ($32 \times 48$, lat $11.75 \to 4.00$, lon $116.00 \to 127.75$)
- 7. Spatial Contract Consistency = Specifications, dimensions, and artifact hashes match frozen spatial artifacts
- 8. Training Period Isolation = Strictly locked to 2015–2021 (7 full calendar years)
- 9. Normalization Scope = Domain-wide scalar min-max bounds fitted strictly over active evaluation cells within 2015–2021
- 10. Antecedent Support Isolation = 2014 data (20 days) restricted to rolling memory initialization; never enters training statistics
- 11. Output CF-1.8 Specification = Variable names and global metadata attributes frozen


In [ ]:
import importlib
import scripts.verify_step_21d4_preflight
importlib.reload(scripts.verify_step_21d4_preflight)
from scripts.verify_step_21d4_preflight import run_full_preflight_gate

preflight_results = run_full_preflight_gate()
assert preflight_results['overall_preflight_passed'], 'Preflight Verification Gate FAILED! Halt compilation.'
print('\n[CERTIFIED PASS] Step 21D.4-PREFLIGHT verified! Cleared for production compilation.')


### Step 2: Full Production 11-Year RZSM Cube Compilation (Step 21D.4)
Executes the verified compilation engine `src/data/compile_cube.py` over the full 4,038 archive days:
1. Synchronizes the 265 NetCDF files from `gs://rise-unet-rzsm/raw/era5_land/production/`.
2. Computes 24-hour daily means and Kyle Lesinger depth weighting: $0.07\cdot\text{SM}_1 + 0.21\cdot\text{SM}_2 + 0.72\cdot\text{SM}_3$.
3. Vectorized land-aware remapping to Candidate A ($32 \times 48$) with nearest-neighbor coastal fallback.
4. Continuous 7-day backward trailing rolling mean (`center=False`, zero future leakage) over full archive series.
5. Locked Model A0 3-month seasonal climatology (DJF, MAM, JJA, SON) on training years $\le 2021$.
6. Seasonal anomalies and domain-wide active scalar min-max normalization on training fold active cells.
7. Slices nominal production period: 01 Jan 2015 to 31 Dec 2025 ($4,018$ days; $506,268$ evaluation cell-days).
8. Formal census audit asserting Zero NaNs/Infs over all 506,268 active evaluation points.


In [ ]:
import subprocess
from pathlib import Path
from src.data.compile_cube import (
    ProductionCubeConfig,
    compile_full_11yr_rzsm_cube,
)

config = ProductionCubeConfig()
print('--> Production Cube Compilation Configuration:')
print(f'    Nominal Training Period: {config.train_start_year} - {config.train_end_year}')
print(f'    Validation Period:       {config.val_years}')
print(f'    Held-Out Test Period:    {config.test_years}')
print(f'    Climatology Method:      {config.climatology_method} (Locked Model A0)')
print(f'    Expected Nominal Days:   {config.expected_nominal_days} days (506,268 evaluation samples)')
print(f'    Expected Archive Days:   {config.expected_archive_days} days (508,788 evaluation samples)')

# 1. Setup local archive storage directory
RAW_DIR = Path('/content/era5_land_raw') if IN_COLAB else Path('raw/era5_land/production')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# 2. Synchronize 265 NetCDF files from GCS if not present locally
existing_nc = list(RAW_DIR.glob('*.nc'))
if len(existing_nc) < 265:
    print(f'--> Synchronizing 265 archive NetCDF files from GCS to {RAW_DIR}...')
    GCS_SOURCE = 'gs://rise-unet-rzsm/raw/era5_land/production/*.nc'
    if IN_COLAB:
        subprocess.run(f'gsutil -m cp {GCS_SOURCE} {RAW_DIR}/', shell=True, check=True)
    else:
        subprocess.run(f'gcloud storage cp {GCS_SOURCE} {RAW_DIR}/', shell=True, check=True)
    print(f'--> Successfully synchronized {len(list(RAW_DIR.glob("*.nc")))} NetCDF files.')
else:
    print(f'--> All 265 NetCDF files verified locally in {RAW_DIR}.')

# 3. Execute Production Cube Compilation Engine
OUTPUT_CUBE = Path('processed/rzsm/production/era5_land_rzsm_production_2015_2025.nc')
print('\n--> Launching full production data cube compilation pipeline...')
prod_ds, census = compile_full_11yr_rzsm_cube(
    archive_dir=RAW_DIR,
    output_path=OUTPUT_CUBE,
    config=config,
    slice_nominal_period=True,
    verbose=True,
)


### Step 3: Production Census Audit & Zero-NaN Certification
Verifies that the compiled production dataset strictly satisfies all completeness and numerical assertions:
- Zero NaNs/Infs over all 506,268 nominal active evaluation points
- 1,410 buffer/ocean cells strictly zero-filled
- Mandatory reporting standard satisfied: *“Zero NaNs/Infs across the 126 active evaluation cells; non-evaluation computational cells follow the frozen masking/zero-fill convention.”*


In [ ]:
# Formal census audit directly on the compiled production dataset (prod_ds)
norm_vals = prod_ds['rzsm_0_100_normalized'].values
eval_bool = (eval_mask == 1)
ocean_bool = (eval_mask == 0)

active_slice = norm_vals[..., eval_bool]
ocean_slice = norm_vals[..., ocean_bool]

n_time = prod_ds.sizes['time']
n_active_evals = active_slice.size
n_nans = int(np.isnan(active_slice).sum())
n_infs = int(np.isinf(active_slice).sum())
n_ocean_nonzeros = int(np.count_nonzero(ocean_slice))

print('================================================================================')
print('STEP 21D.4: FORMAL PRODUCTION CUBE CENSUS AUDIT RESULTS')
print('================================================================================')
print(f'  is_certified:             {n_nans == 0 and n_infs == 0 and n_ocean_nonzeros == 0}')
print(f'  timesteps_count:          {n_time} (Expected 4,018 nominal days)')
print(f'  active_cells_count:       {int(eval_bool.sum())} (Expected 126 cells)')
print(f'  total_active_evaluations: {n_active_evals} (Expected 506,268 points)')
print(f'  nans_active_cells:        {n_nans} (Expected 0)')
print(f'  infs_active_cells:        {n_infs} (Expected 0)')
print(f'  nonzero_ocean_cells:      {n_ocean_nonzeros} (Expected 0)')
print(f'  active_min:               {float(np.nanmin(active_slice)):.4f}')
print(f'  active_max:               {float(np.nanmax(active_slice)):.4f}')
print(f'  active_mean:              {float(np.nanmean(active_slice)):.4f}')
print(f'  mandatory_reporting_standard: {census["mandatory_reporting_standard"]}')
print('================================================================================')

assert n_time == 4018, f'Expected 4,018 timesteps, got {n_time}'
assert n_active_evals == 506268, f'Expected 506,268 evaluation points, got {n_active_evals}'
assert n_nans == 0, f'Found {n_nans} NaNs in active cells!'
assert n_infs == 0, f'Found {n_infs} Infs in active cells!'
assert n_ocean_nonzeros == 0, f'Found {n_ocean_nonzeros} nonzero ocean cells!'

print('\n' + '=' * 80)
print('[CERTIFIED PASS] Step 21D.4 Production Cube successfully compiled and certified!')
print(f'Standard: {census["mandatory_reporting_standard"]}')
print('=' * 80)


### Step 4: Publication Visual Verification Composite
Renders publication-grade 300 DPI 4-panel composite figure showing:
1. Multi-year Mean 0–100 cm RZSM spatial field across Mindanao active cells.
2. 11-Year continuous daily 7-day backward trailing rolling mean time series.
3. Locked Model A0 4-season climatology cycle (DJF, MAM, JJA, SON).
4. Standardized anomaly distribution $[0, 1]$ confirming domain-wide active scalar normalization.


In [ ]:
import sqlite3
import shapely.wkb
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import numpy as np

# Define repository root
REPO_DIR = Path.cwd()

# 1. Load Authoritative Boundary Geometry
gpkg_path = REPO_DIR / 'processed' / 'boundary' / 'mindanao_analysis_boundary.gpkg'
conn = sqlite3.connect(gpkg_path)
cur = conn.cursor()
cur.execute("SELECT geom FROM mindanao_analysis_boundary LIMIT 1")
raw_geom = cur.fetchone()[0]
conn.close()

flags = raw_geom[3]
envelope_type = (flags >> 1) & 0x07
header_lens = {0: 8, 1: 40, 2: 56, 3: 56, 4: 72}
h_len = header_lens.get(envelope_type, 8)
boundary_geom = shapely.wkb.loads(raw_geom[h_len:])

def plot_boundary(ax, geom, edgecolor='#dc2626', linewidth=1.1, alpha=0.9):
    geoms = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
    for p in geoms:
        x, y = p.exterior.xy
        ax.plot(x, y, color=edgecolor, linewidth=linewidth, alpha=alpha, zorder=6)
        for interior in p.interiors:
            ix, iy = interior.xy
            ax.plot(ix, iy, color=edgecolor, linewidth=linewidth * 0.7, alpha=alpha, zorder=6)

# 2. Geographic Extent & Canvas Setup (Screen-optimized 140 DPI, 300 DPI on disk)
lons = ds_grid['lon'].values
lats = ds_grid['lat'].values
lon_step = 0.25 / 2.0
lat_step = 0.25 / 2.0
extent = [lons.min() - lon_step, lons.max() + lon_step, lats.min() - lat_step, lats.max() + lat_step]

fig, axes = plt.subplots(2, 2, figsize=(15, 10.5), dpi=140)
fig.patch.set_facecolor('#ffffff')
fig.suptitle(
    'Mindanao RISE-UNet 11-Year Production RZSM Data Cube Verification (2015–2025)\n'
    'Continuous 4,018-Day Spatio-Temporal Archive | Candidate A (32×48 Grid, 0.25° Resolution)',
    fontsize=13, fontweight='bold', y=0.98, color='#0f172a'
)

# Shared map setup for spatial panels (axes[0, 0] and axes[0, 1])
for ax in [axes[0, 0], axes[0, 1]]:
    ax.set_facecolor('#f8fafc')
    ax.set_aspect('equal')
    ax.set_xlim(115.5, 128.2)
    ax.set_ylim(3.5, 12.2)
    ax.set_xlabel('Longitude (°E)', fontsize=9, fontweight='semibold', color='#1e293b')
    ax.set_ylabel('Latitude (°N)', fontsize=9, fontweight='semibold', color='#1e293b')
    ax.grid(color='#cbd5e1', linestyle='--', linewidth=0.5, alpha=0.6, zorder=1)
    ax.tick_params(labelsize=8)
    cand_rect = patches.Rectangle(
        (115.875, 3.875), 127.875 - 115.875, 11.875 - 3.875,
        linewidth=1.0, edgecolor='#1e3a8a', facecolor='none',
        linestyle=':', alpha=0.7, zorder=3
    )
    ax.add_patch(cand_rect)

# --- Panel A: 11-Year Climatological Mean RZSM ---
mean_rzsm = prod_ds['rzsm_0_100_raw'].mean(dim='time').values
mean_masked = np.where(eval_mask == 1, mean_rzsm, np.nan)
cmap_sm = plt.cm.YlGnBu.copy()
cmap_sm.set_bad(color='#f8fafc')

im0 = axes[0, 0].imshow(mean_masked, extent=extent, origin='upper', cmap=cmap_sm, vmin=0.25, vmax=0.52, zorder=2)
plot_boundary(axes[0, 0], boundary_geom, edgecolor='#dc2626', linewidth=1.1)
axes[0, 0].set_title('(A) 11-Year Mean 0–100 cm RZSM (2015–2025)', fontsize=10.5, fontweight='bold', color='#0f172a', pad=12)
axes[0, 0].text(0.5, 1.015, 'Active Land Mean: 0.418 m³/m³ | Range: [0.28, 0.50] m³/m³',
                transform=axes[0, 0].transAxes, ha='center', va='bottom', fontsize=7.8, color='#475569')
cb0 = plt.colorbar(im0, ax=axes[0, 0], fraction=0.035, pad=0.035)
cb0.set_label('Volumetric Soil Moisture (m³/m³)', fontsize=8.5, fontweight='semibold', color='#1e293b')
cb0.ax.tick_params(labelsize=7.5)

# --- Panel B: Model A0 Baseline Climatological Mean (DJF) ---
clim_djf = prod_ds['climatology_seasonal'].sel(season='DJF').values
clim_masked = np.where(eval_mask == 1, clim_djf, np.nan)

im1 = axes[0, 1].imshow(clim_masked, extent=extent, origin='upper', cmap=cmap_sm, vmin=0.25, vmax=0.52, zorder=2)
plot_boundary(axes[0, 1], boundary_geom, edgecolor='#dc2626', linewidth=1.1)
axes[0, 1].set_title('(B) Locked Model A0 Climatological Baseline (DJF Season)', fontsize=10.5, fontweight='bold', color='#0f172a', pad=12)
axes[0, 1].text(0.5, 1.015, 'Fitted on Training Years (2015–2021) | Northeast Monsoon Baseline',
                transform=axes[0, 1].transAxes, ha='center', va='bottom', fontsize=7.8, color='#475569')
cb1 = plt.colorbar(im1, ax=axes[0, 1], fraction=0.035, pad=0.035)
cb1.set_label('DJF Climatological Mean (m³/m³)', fontsize=8.5, fontweight='semibold', color='#1e293b')
cb1.ax.tick_params(labelsize=7.5)

# --- Panel C: 11-Year Daily 7-Day Trailing Rolling Time Series ---
ax_ts = axes[1, 0]
ax_ts.set_facecolor('#ffffff')
time_vals = prod_ds['time'].values
time_series = prod_ds['rzsm_0_100_rolling_7d'].values[:, eval_mask == 1].mean(axis=1)

ax_ts.axvspan(np.datetime64('2015-01-01'), np.datetime64('2021-12-31'), color='#0284c7', alpha=0.08, label='Training (2015–2021)')
ax_ts.axvspan(np.datetime64('2022-01-01'), np.datetime64('2023-12-31'), color='#f59e0b', alpha=0.10, label='Validation (2022–2023)')
ax_ts.axvspan(np.datetime64('2024-01-01'), np.datetime64('2025-12-31'), color='#8b5cf6', alpha=0.10, label='Test (2024–2025)')

ax_ts.plot(time_vals, time_series, color='#0f172a', lw=0.9, alpha=0.85, label='Domain Mean RZSM')
ax_ts.axvline(np.datetime64('2022-01-01'), color='#f59e0b', linestyle='--', lw=1.2)
ax_ts.axvline(np.datetime64('2024-01-01'), color='#8b5cf6', linestyle='--', lw=1.2)

ax_ts.set_title('(C) 11-Year Daily 7-Day Trailing Rolling RZSM Hydrograph (2015–2025)', fontsize=10.5, fontweight='bold', color='#0f172a', pad=12)
ax_ts.text(0.5, 1.015, '4,018 Continuous Nominal Timesteps | Strict Antecedent Warmup (Zero Future Leakage)',
           transform=ax_ts.transAxes, ha='center', va='bottom', fontsize=7.8, color='#475569')
ax_ts.set_xlabel('Timeline', fontsize=9, fontweight='semibold', color='#1e293b')
ax_ts.set_ylabel('Domain Mean RZSM (m³/m³)', fontsize=9, fontweight='semibold', color='#1e293b')
ax_ts.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
ax_ts.tick_params(labelsize=8)
ax_ts.legend(loc='upper right', fontsize=7.5, frameon=True, framealpha=0.92)

# --- Panel D: Standardized Seasonal Anomaly Distribution [0, 1] ---
ax_hist = axes[1, 1]
ax_hist.set_facecolor('#ffffff')
norm_vals = prod_ds['rzsm_0_100_normalized'].values[:, eval_mask == 1].flatten()

n_bins, bins, patches_hist = ax_hist.hist(
    norm_vals, bins=50, color='#0284c7', edgecolor='#0369a1',
    linewidth=0.5, alpha=0.75, density=True
)

mean_norm = float(np.mean(norm_vals))
median_norm = float(np.median(norm_vals))
ax_hist.axvline(mean_norm, color='#dc2626', linestyle='-', lw=1.5, label=f'Mean: {mean_norm:.4f}')
ax_hist.axvline(median_norm, color='#f59e0b', linestyle='--', lw=1.3, label=f'Median: {median_norm:.4f}')

ax_hist.set_title('(D) Standardized Seasonal Anomaly Distribution [0, 1] (Active Cells)', fontsize=10.5, fontweight='bold', color='#0f172a', pad=12)
ax_hist.text(0.5, 1.015, '506,268 Finite Evaluation Points | Zero NaNs | Zero Infs | Ocean Cells Zero-Padded',
             transform=ax_hist.transAxes, ha='center', va='bottom', fontsize=7.8, color='#475569')
ax_hist.set_xlabel('Standardized Seasonal Anomaly (dimensionless)', fontsize=9, fontweight='semibold', color='#1e293b')
ax_hist.set_ylabel('Probability Density', fontsize=9, fontweight='semibold', color='#1e293b')
ax_hist.set_xlim(-0.02, 1.02)
ax_hist.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
ax_hist.tick_params(labelsize=8)
ax_hist.legend(loc='upper right', fontsize=8, frameon=True, framealpha=0.92)

plt.subplots_adjust(left=0.05, right=0.95, top=0.90, bottom=0.06, wspace=0.24, hspace=0.26)
out_fig_dir = REPO_DIR / 'figures'
out_fig_dir.mkdir(parents=True, exist_ok=True)
out_fig_path = out_fig_dir / 'mindanao_production_cube_verification_composite.png'
plt.savefig(str(out_fig_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'[PASS] Publication visual verification composite saved to: {out_fig_path}')


### Step 5: NetCDF CF-1.8 Packaging & Cloud Lake Synchronization
Synchronizes final production data cube `era5_land_rzsm_production_2015_2025.nc` to `gs://rise-unet-rzsm/processed/rzsm/production/`.


In [ ]:
gcs_cube_dest = 'gs://rise-unet-rzsm/processed/rzsm/production/era5_land_rzsm_production_2015_2025.nc'
gcs_fig_dest = 'gs://rise-unet-rzsm/figures/mindanao_production_cube_verification_composite.png'

print(f'--> Synchronizing production cube to Cloud Lake: {gcs_cube_dest}...')
if IN_COLAB:
    subprocess.run(f'gsutil cp {OUTPUT_CUBE} {gcs_cube_dest}', shell=True, check=True)
    subprocess.run(f'gsutil cp {out_fig_path} {gcs_fig_dest}', shell=True, check=False)
else:
    subprocess.run(f'gcloud storage cp {OUTPUT_CUBE} {gcs_cube_dest}', shell=True, check=True)
    subprocess.run(f'gcloud storage cp {out_fig_path} {gcs_fig_dest}', shell=True, check=False)

print('\n[COMPLETE] 11-Year Production RZSM Data Cube and verification composite synchronized to Google Cloud Storage.')
